## **2.1 Dataset Overview**
Load the SmartCare dataset and inspect its structure before any analysis.

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/smartcare_ai_dataset_1000.csv')
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Shape: 1000 rows, 33 columns


,record_id,patient_id,age,gender,blood_group,department,diagnosis,appointment_date,waiting_days,previous_appointments,...,consultation_fee_lkr,room_charge_lkr,lab_charge_lkr,medicine_charge_lkr,total_bill_lkr,payment_status,payment_method,no_show,readmitted_30_days,disease_risk_level
0,1,P10001,53,Male,A-,General Medicine,Migraine,2025-04-10,10,1,...,2000,0,0,11596,13596,Paid,Insurance,0,0,High
1,2,P10002,26,Male,B-,General Medicine,Diabetes,2025-05-15,2,3,...,2000,0,0,3652,5652,Paid,Insurance,0,0,Medium
2,3,P10003,22,Male,B+,Orthopedics,Back Pain,2025-07-09,22,7,...,2500,0,1200,2562,6262,Unpaid,Insurance,1,0,Medium
3,4,P10004,44,Female,AB-,Cardiology,Asthma,2025-10-16,16,1,...,2000,0,5000,10262,17262,Paid,Online,0,0,Medium
4,5,P10005,51,Female,O+,Neurology,Hypertension,2025-12-18,12,4,...,4000,0,6000,10414,20414,Paid,Cash,0,0,Medium


In [2]:
df.dtypes

record_id                         int64
patient_id                       object
age                               int64
gender                           object
blood_group                      object
department                       object
diagnosis                        object
appointment_date                 object
waiting_days                      int64
previous_appointments             int64
missed_previous_appointments      int64
appointment_status               object
admitted                          int64
room_type                        object
length_of_stay_days               int64
previous_admissions               int64
systolic_bp                       int64
diastolic_bp                      int64
blood_sugar_mg_dl                 int64
cholesterol_mg_dl                 int64
bmi                             float64
lab_tests_count                   int64
treatments_count                  int64
consultation_fee_lkr              int64
room_charge_lkr                   int64


## **2.2 Data Dictionary Interpretation**
Load the official data dictionary. All attribute meanings used in this notebook come directly from this file — no interpretation beyond what is documented.

In [3]:
data_dict = pd.read_csv('../data/raw/smartcare_ai_dataset_data_dictionary.csv')
data_dict

,Column,Description
0,record_id,Unique row identifier
1,patient_id,Synthetic patient identifier
2,age,Patient age in years
3,gender,Patient gender
4,blood_group,Patient blood group
5,department,Hospital department
6,diagnosis,Primary diagnosis category
7,appointment_date,Appointment date
8,waiting_days,Number of days between booking and appointment
9,previous_appointments,Number of previous appointments


## **2.3 Attribute Description & Logical Feature Groups**
Columns are grouped by category, per the coursework specification, for easier reasoning during preprocessing and modelling.

In [4]:
feature_groups = {
    'Patient Info': ['patient_id', 'age', 'gender', 'blood_group'],
    'Clinical': ['diagnosis', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl',
                 'cholesterol_mg_dl', 'bmi'],
    'Appointment/Operations': ['department', 'appointment_date', 'waiting_days',
                                'previous_appointments', 'missed_previous_appointments',
                                'appointment_status'],
    'Admission': ['admitted', 'room_type', 'length_of_stay_days', 'previous_admissions',
                  'lab_tests_count', 'treatments_count'],
    'Financial': ['consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr',
                  'medicine_charge_lkr', 'total_bill_lkr', 'payment_status', 'payment_method'],
    'Identifiers': ['record_id', 'patient_id'],
    'Targets': ['no_show', 'readmitted_30_days', 'disease_risk_level']
}

for group, cols in feature_groups.items():
    print(f"\n{group}:")
    print(df[cols].dtypes)


Patient Info:
patient_id     object
age             int64
gender         object
blood_group    object
dtype: object

Clinical:
diagnosis             object
systolic_bp            int64
diastolic_bp           int64
blood_sugar_mg_dl      int64
cholesterol_mg_dl      int64
bmi                  float64
dtype: object

Appointment/Operations:
department                      object
appointment_date                object
waiting_days                     int64
previous_appointments            int64
missed_previous_appointments     int64
appointment_status              object
dtype: object

Admission:
admitted                int64
room_type              object
length_of_stay_days     int64
previous_admissions     int64
lab_tests_count         int64
treatments_count        int64
dtype: object

Financial:
consultation_fee_lkr     int64
room_charge_lkr          int64
lab_charge_lkr           int64
medicine_charge_lkr      int64
total_bill_lkr           int64
payment_status          object
payment

## **2.4 Predictors / Identifiers / Alternative-Target Distinction**
This project uses `disease_risk_level` as the sole target (Option C). `no_show` and 
`readmitted_30_days` are alternative targets documented in the data dictionary for other 
coursework options; they are excluded from the feature set for this task and not analyzed 
further in this notebook.

In [5]:
identifiers = ['record_id', 'patient_id']
target = 'disease_risk_level'
alternative_targets = ['no_show', 'readmitted_30_days']

candidate_predictors = [c for c in df.columns
                         if c not in identifiers + alternative_targets + [target]]

print(f"Identifiers ({len(identifiers)}): {identifiers}")
print(f"Alternative targets excluded ({len(alternative_targets)}): {alternative_targets}")
print(f"Target: {target}")
print(f"Candidate predictors ({len(candidate_predictors)}): {candidate_predictors}")

Identifiers (2): ['record_id', 'patient_id']
Alternative targets excluded (2): ['no_show', 'readmitted_30_days']
Target: disease_risk_level
Candidate predictors (28): ['age', 'gender', 'blood_group', 'department', 'diagnosis', 'appointment_date', 'waiting_days', 'previous_appointments', 'missed_previous_appointments', 'appointment_status', 'admitted', 'room_type', 'length_of_stay_days', 'previous_admissions', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi', 'lab_tests_count', 'treatments_count', 'consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr', 'medicine_charge_lkr', 'total_bill_lkr', 'payment_status', 'payment_method']


## **2.5 Target Variable Identification & Distribution**
`disease_risk_level` is a multi-class target with classes Low, Medium, High, as specified 
in the coursework. Class distribution is computed programmatically below.

In [6]:
target_counts = df[target].value_counts()
target_pct = df[target].value_counts(normalize=True) * 100

pd.DataFrame({'count': target_counts, 'percentage': target_pct.round(2)})

,count,percentage
disease_risk_level,,
Medium,469,46.9
High,400,40.0
Low,131,13.1


## **2.6 Initial Data Quality Assessment**
Identify data-quality issues only — no fixes are applied here; that belongs to Task 03.

In [7]:
# Missing values
missing = df.isnull().sum()
missing[missing > 0]

room_type    906
dtype: int64

In [8]:
# Duplicate records
print(f"Duplicate rows: {df.duplicated().sum()}")

Duplicate rows: 0


In [9]:
# Observe whether room_type missingness aligns with admitted status
pd.crosstab(df['admitted'], df['room_type'].isnull(), rownames=['admitted'], colnames=['room_type_missing'])

room_type_missing,False,True
admitted,,
0,0,670
1,94,236


## **2.7 Known Limitations / Unknowns**

- **Synthetic data**: The data dictionary explicitly labels several fields (e.g. `systolic_bp`, 
  `diastolic_bp`, `blood_sugar_mg_dl`, `cholesterol_mg_dl`, `bmi`, `patient_id`) as "Synthetic," 
  meaning this dataset does not reflect real patient records.
- **Target-generation mechanism undocumented**: Neither the coursework specification nor the 
  data dictionary explains how `disease_risk_level` was derived from the underlying clinical 
  attributes. Any relationship observed between predictors and this target in later tasks should 
  be treated as an empirical pattern in this dataset, not a validated clinical rule.
- **Alternative targets show potential leakage**: `no_show` and `readmitted_30_days` (not used 
  in this task) were observed to align closely with `appointment_status` and `admitted` 
  respectively. This is noted here for awareness only, as this notebook does not model those targets.
- **Missingness cause not confirmed**: The observed relationship between `admitted` and missing 
  `room_type` (Section 2.6) is a pattern in the data, not a confirmed business rule — no source 
  document states this explicitly.
- **No documentation on data collection process**: Nothing in the supplied files describes how 
  or when the data was collected, so representativeness cannot be assessed.